# Leave Cluster Out

`LeaveClusterOut` implements spatial block cross-validation using a
user-supplied clustering estimator. Each fold holds out one spatial cluster
as the test set; the training set is all observations in every other cluster.

This is the leave-one-block-out analogue of `ClusterStratifiedKFold`. Unlike
DGGS-based blocking, the cluster shapes adapt to the spatial density of the
data. HDBSCAN also produces *noise points* (label = -1) — observations too
isolated to belong to any cluster — and `LeaveClusterOut` offers three ways to
handle them:

- **`train_only`** (default): noise points appear in every training set, never test.
- **`drop`**: noise points are excluded from both train and test.
- **`nearest`**: each noise point is reassigned to the cluster of its nearest
  non-noise neighbour and then participates in that cluster's fold as a regular member.

In [ ]:
import geodatasets
import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import HDBSCAN

from spml.validation import LeaveClusterOut

In [ ]:
gdf = gpd.read_file(geodatasets.get_path("geoda.home_sales")).sample(
    10000, random_state=0, ignore_index=True
)
print(str(len(gdf)) + " house sales")

## Fit HDBSCAN

Using `metric='haversine'` lets HDBSCAN work directly in geographic space
without reprojecting. Haversine expects coordinates as `(lat, lon)` in radians.

We pre-fit the clusterer and pass it to `LeaveClusterOut`. When `labels_` is
already present on the estimator, `split()` uses those labels directly and
does not refit — useful for comparing `noise` options without redundant work.

The scan below finds the largest `min_cluster_size` that still yields at
least 8 clusters.

In [ ]:
# haversine expects (lat, lon) in radians
coords_rad = np.radians(gdf.get_coordinates()[["y", "x"]])

best_mcs = 100
for mcs in [50, 100, 150, 200, 250, 300]:
    labels_ = (
        HDBSCAN(min_cluster_size=mcs, metric="haversine", copy=False)
        .fit(coords_rad)
        .labels_
    )
    n = (np.unique(labels_) >= 0).sum()
    n_noise = (labels_ == -1).sum()
    print(
        "  min_cluster_size="
        + str(mcs)
        + " -> "
        + str(n)
        + " clusters, "
        + str(n_noise)
        + " noise ("
        + str(round(100 * n_noise / len(labels_), 1))
        + "%)"
    )
    if n >= 8:
        best_mcs = mcs

print("\nUsing min_cluster_size=" + str(best_mcs))
hdb = HDBSCAN(min_cluster_size=best_mcs, metric="haversine", copy=False).fit(coords_rad)

## Cluster assignments

In [ ]:
# Pass the pre-fitted HDBSCAN; LeaveClusterOut detects labels_ and skips refitting
lco = LeaveClusterOut(hdb, noise="train_only")
splits_to = list(lco.split(gdf))

labels = lco.labels_
n_noise = (labels == -1).sum()
print("Clusters (folds) : " + str(lco.n_clusters_))
print(
    "Noise points     : "
    + str(n_noise)
    + " ("
    + str(round(100 * n_noise / len(gdf), 1))
    + "%)"
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
noise_mask = labels == -1

cluster_labels = labels[~noise_mask]
gdf[~noise_mask].plot(
    ax=ax, column=cluster_labels, markersize=1.5, zorder=2, categorical=True
)
gdf[noise_mask].plot(
    ax=ax,
    color="lightgrey",
    markersize=0.5,
    zorder=1,
)
legend_labels = ["cluster " + str(i) for i in range(lco.n_clusters_)]
legend_labels.append("noise (" + str(n_noise) + ")")
ax.legend(labels=legend_labels, loc="lower left", fontsize=8)
ax.set_title(
    "HDBSCAN -- "
    + str(lco.n_clusters_)
    + " clusters, "
    + str(n_noise)
    + " noise points (min_cluster_size="
    + str(best_mcs)
    + ")"
)
plt.tight_layout()

## Noise point options: `train_only`, `drop`, and `nearest`

Noise points are never the sole subject of a test fold. The `noise` parameter
controls how they participate in each fold:

- **`train_only`** (default): noise points are in every training set.
  The model sees them during training but is never evaluated on them.
- **`drop`**: noise points are excluded from both train and test.
  Use this when nearby noise points might leak spatial information
  about the held-out cluster.
- **`nearest`**: each noise point is reassigned to the cluster of its
  nearest non-noise neighbour. It then acts as a full member of that cluster —
  appearing in the test fold when that cluster is held out, and in the training
  fold otherwise. The original labels (including `-1`) are preserved in `labels_`.

Because `hdb` is already fitted (has `labels_`), none of the calls below refit.

In [ ]:
lco_drop = LeaveClusterOut(lco.clusterer_, noise="drop")
lco_nearest = LeaveClusterOut(lco.clusterer_, noise="nearest")

splits_drop = list(lco_drop.split(gdf))
splits_nearest = list(lco_nearest.split(gdf))

stats = []
for i, ((tr_to, te), (tr_dr, _), (tr_nr, _)) in enumerate(
    zip(splits_to, splits_drop, splits_nearest, strict=False)
):
    stats.append(
        {
            "fold": i,
            "test size": len(te),
            "train (train_only)": len(tr_to),
            "train (drop)": len(tr_dr),
            "train (nearest)": len(tr_nr),
            "noise excluded (drop)": len(tr_to) - len(tr_dr),
        }
    )
pd.DataFrame(stats)

## Noise handling visualised

The three panels below show the same fold under each noise option.
Orange points on the left are the noise points that `train_only` keeps in training.
In the `nearest` panel, noise points assigned to the held-out cluster appear in the
test set (red) alongside the regular cluster members.

In [ ]:
fold = 0
tr_to, te = splits_to[fold]
tr_dr, _ = splits_drop[fold]
tr_nr, te_nr = splits_nearest[fold]

noise_in_train = np.setdiff1d(tr_to, tr_dr)  # noise visible in train_only
noise_in_test_nr = np.intersect1d(
    te_nr, np.where(lco.labels_ == -1)[0]
)  # noise -> test in nearest

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)

configs = [
    (tr_to, te, "noise='train_only'"),
    (tr_dr, te, "noise='drop'"),
    (tr_nr, te_nr, "noise='nearest'"),
]

for ax, (train, test, title) in zip(axes, configs, strict=False):
    gdf.iloc[train].plot(
        ax=ax,
        color="steelblue",
        markersize=0.4,
        alpha=0.4,
    )
    gdf.iloc[test].plot(ax=ax, color="firebrick", markersize=3, zorder=3)
    ax.set_title(title, fontsize=12)
    ax.set_axis_off()

# highlight noise points on the train_only panel
gdf.iloc[noise_in_train].plot(
    ax=axes[0], color="orange", markersize=1.5, zorder=2, alpha=0.8
)

# highlight reassigned noise in the nearest panel (split between test and train)
noise_in_train_nr = np.intersect1d(tr_nr, np.where(lco.labels_ == -1)[0])

gdf.iloc[noise_in_test_nr].plot(
    ax=axes[2],
    color="darkorange",
    markersize=4,
    zorder=4,
    alpha=0.9,
)
gdf.iloc[noise_in_train_nr].plot(
    ax=axes[2],
    color="orange",
    markersize=1.5,
    zorder=2,
    alpha=0.8,
)

train_patch = mpatches.Patch(color="steelblue", label="train (cluster pts)")
test_patch = mpatches.Patch(color="firebrick", label="test cluster")
noise_train_patch = mpatches.Patch(color="orange", label="noise -> train")
noise_test_patch = mpatches.Patch(color="darkorange", label="noise -> test (nearest)")

axes[0].legend(
    handles=[train_patch, test_patch, noise_train_patch], loc="lower left", fontsize=8
)
axes[1].legend(handles=[train_patch, test_patch], loc="lower left", fontsize=8)
axes[2].legend(
    handles=[train_patch, test_patch, noise_train_patch, noise_test_patch],
    loc="lower left",
    fontsize=8,
)

fig.suptitle(
    "LeaveClusterOut fold " + str(fold) + ": noise handling comparison", fontsize=13
)
plt.tight_layout()